# Self-supervised vision — learn image features without labels, then probe

_Capstones_

**Three contrastive papers, one system: pretrain on unlabelled images, freeze, and let a linear probe beat a from-scratch net when labels are scarce.**

---

This notebook builds the system one step at a time: first we look at MNIST and the **two random views** SimCLR trains on, then we build the NT-Xent contrastive loss tensor by tensor, define the encoder and projection head, pretrain on **unlabelled** MNIST, freeze the encoder, and finally pit a linear probe against a from-scratch net across label budgets.

The paper trail is the contrastive-learning family: **SimCLR** makes two augmented views of one image the positive pair; **MoCo** shows how a memory/queue can provide many negatives; **CLIP** scales the same match-the-pair idea to image-text pairs. Here we keep the core SimCLR mechanism small enough to run and inspect.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders and dropdowns; in a plain Python run they fall back to sensible defaults. Run top to bottom, experiment with the knobs, then tackle the practice prompts at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib / torch / torchvision ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

# Keep the original lesson seeds.
torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## First, look at the data

Before any model or loss, inspect the real images. MNIST examples include labels, but in this notebook those labels are **ignored during SimCLR pretraining** and used only later for the linear-probe evaluation and result visualizations.

### Step 1 — Load MNIST and make the unlabelled subset

The pretraining pool is the original 3000-image subset selected with `np.random.RandomState(0)`. We keep the labels alongside the images only so the later probe and plots can ask, "did useful class structure emerge?"

In [ ]:
base = T.ToTensor()
raw = torchvision.datasets.MNIST(root="./data", train=True, download=True)
idx = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])  # used ONLY later, never in SimCLR pretraining

print("dataset: MNIST train")
print("full dataset samples:", len(raw))
print("unlabelled pretraining subset:", len(imgs))
print("one PIL image size:", imgs[0].size, " label carried for later probe:", int(labels[0]))

### Step 2 — A tiny table of real samples

Images are not tabular rows, but a table can still summarize useful facts: the original dataset index, the held-out digit label, the image shape, and simple pixel statistics. The `mean_pixel` column is computed from the actual tensor values in `[0, 1]`.

In [ ]:
rows = []
for k in range(12):
    x = base(imgs[k])
    rows.append({
        "subset_pos": k,
        "mnist_index": int(idx[k]),
        "label_for_later": int(labels[k]),
        "tensor_shape": tuple(x.shape),
        "mean_pixel": round(float(x.mean()), 4),
        "nonzero_pixels": int((x > 0).sum()),
    })
sample_df = pd.DataFrame(rows)
display(sample_df)

### Step 3 — Class balance (for later probing only)

The unlabelled subset was sampled randomly, so the digits are roughly balanced. SimCLR does **not** read these labels; the bar chart simply tells us whether the later low-label probe is drawing from a reasonable mix of digits.

In [ ]:
counts = torch.bincount(labels, minlength=10).numpy()
balance_df = pd.DataFrame({"digit": np.arange(10), "count_in_subset": counts})
display(balance_df)

plt.figure(figsize=(7, 3))
plt.bar(balance_df["digit"], balance_df["count_in_subset"], color="#79c0ff")
plt.xticks(range(10)); plt.xlabel("digit label (held out during pretrain)"); plt.ylabel("count")
plt.title("MNIST subset label balance")
plt.show()

### Step 4 — Look at the pixels

The sample grid shows the raw images. The histogram shows that most pixels are near black, with a smaller set of bright stroke pixels — useful context for why random crops and blur still preserve the digit identity.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(8, 3))
for ax, image, label in zip(axes.ravel(), imgs[:12], labels[:12]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"label {int(label)}")
    ax.axis("off")
plt.suptitle("MNIST examples")
plt.tight_layout()
plt.show()

pix = torch.stack([base(im) for im in imgs[:min(256, len(imgs))]]).flatten().numpy()
plt.figure(figsize=(6, 3))
plt.hist(pix, bins=30, color="#c89bff")
plt.xlabel("pixel intensity"); plt.ylabel("count")
plt.title("pixel distribution")
plt.show()

### Step 5 — The two-view augmentation (the positive pair)

This is SimCLR's key data trick. For one original image, we sample **two independent augmented views**. Those two views are treated as the only positive pair in the batch; every other image's views are negatives. The label is still not used.

In [ ]:
aug = T.Compose([
    T.RandomResizedCrop(28, scale=(0.5, 1.0)),
    T.RandomApply([T.GaussianBlur(3)], 0.5),
    T.ToTensor(),
])

pick = 0
torch.manual_seed(1)
v1 = aug(imgs[pick])
torch.manual_seed(2)
v2 = aug(imgs[pick])
orig = base(imgs[pick])

fig, axes = plt.subplots(1, 3, figsize=(7, 2.4))
for ax, tensor, title in zip(axes, [orig, v1, v2], ["original", "view 1", "view 2"]):
    ax.imshow(tensor.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title); ax.axis("off")
plt.suptitle("one image -> two random positive views")
plt.tight_layout(); plt.show()
print("label exists for later:", int(labels[pick]), "  label used in this augmentation step: NO")

**🎛️ Play with it — augmentation strength**

The augmentation must be strong enough that the model learns invariances, but not so strong that the digit identity disappears. **Try:** lower the crop scale or toggle blur. **Watch:** the output view change. **Why it matters:** SimCLR learns what stays stable across the two views.

In [ ]:
def aug_strength(image_id=0, min_crop=0.5, blur=True):
    image_id = int(np.clip(image_id, 0, len(imgs) - 1))
    transform = T.Compose([
        T.RandomResizedCrop(28, scale=(float(min_crop), 1.0)),
        T.RandomApply([T.GaussianBlur(3)], 1.0 if blur else 0.0),
        T.ToTensor(),
    ])
    torch.manual_seed(4)
    view = transform(imgs[image_id])
    fig, axes = plt.subplots(1, 2, figsize=(5, 2.4))
    axes[0].imshow(imgs[image_id], cmap="gray"); axes[0].set_title("original"); axes[0].axis("off")
    axes[1].imshow(view.squeeze(), cmap="gray", vmin=0, vmax=1); axes[1].set_title("augmented"); axes[1].axis("off")
    plt.suptitle("augmentation view")
    plt.tight_layout(); plt.show()
    print("label loaded for later =", int(labels[image_id]), "| used now = NO")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(aug_strength,
             image_id=IntSlider(value=0, min=0, max=len(imgs)-1, step=1),
             min_crop=FloatSlider(value=0.5, min=0.2, max=0.9, step=0.05),
             blur=Checkbox(value=True))
except Exception:
    aug_strength()

## Reference implementation — PyTorch

We will build the SimCLR system in small pieces:

| Paper/system | Contributes | In our code |
|---|---|---|
| SimCLR | two stochastic views + NT-Xent | `aug`, `nt_xent(z)` |
| MoCo | many negatives help contrastive learning | our batch negatives stand in for a queue |
| CLIP | match paired views across modalities | same "positive pair vs negatives" softmax idea |

The hero operation is **NT-Xent**, so we build it by hand before packaging it into a function.

### Step 6 — Hero op setup: a tiny batch of embeddings

Suppose a batch contains `N=4` source images. SimCLR stacks the first augmented view of all `N` images, then the second augmented view of all `N` images, giving `2N` embeddings. Row `i`'s positive partner is row `i+N` (or `i-N`).

In [ ]:
torch.manual_seed(0)
N_demo, D_demo = 4, 4
centres = F.normalize(torch.randn(N_demo, D_demo), dim=1)
z_demo = torch.cat([
    centres + 0.08 * torch.randn(N_demo, D_demo),
    centres + 0.08 * torch.randn(N_demo, D_demo),
], dim=0)
view_names = [f"img{k}-v1" for k in range(N_demo)] + [f"img{k}-v2" for k in range(N_demo)]
pos_target = torch.cat([torch.arange(N_demo) + N_demo, torch.arange(N_demo)])

partner_df = pd.DataFrame({"row": range(2*N_demo), "embedding": view_names, "positive_row": pos_target.tolist(),
                           "positive_embedding": [view_names[i] for i in pos_target.tolist()]})
display(partner_df)
print("raw stacked embedding tensor:", tuple(z_demo.shape))

### Step 7 — NT-Xent part A: L2-normalize

Cosine similarity is just a dot product after L2-normalization. Each row is scaled to length 1, so the similarity matrix will measure direction rather than vector magnitude.

In [ ]:
z_norm = F.normalize(z_demo, dim=1)
norm_df = pd.DataFrame({"row": view_names, "norm before": z_demo.norm(dim=1).round(decimals=3).tolist(),
                        "norm after": z_norm.norm(dim=1).round(decimals=3).tolist()})
display(norm_df)

### Step 8 — NT-Xent part B: cosine-similarity matrix

Now compute every pairwise cosine similarity: a `2N x 2N` heatmap. The diagonal is trivially 1 because each view matches itself, so NT-Xent will remove it before the softmax.

In [ ]:
sim_demo = z_norm @ z_norm.t()
plt.figure(figsize=(6, 5))
plt.imshow(sim_demo, cmap="viridis", vmin=-1, vmax=1)
plt.xticks(range(2*N_demo), view_names, rotation=45, ha="right")
plt.yticks(range(2*N_demo), view_names)
plt.title("cosine similarity matrix")
plt.colorbar(label="cosine")
plt.tight_layout(); plt.show()

### Step 9 — NT-Xent part C: mark the positive pairs

The positives are **off the diagonal**: `img0-v1` should match `img0-v2`, `img1-v1` should match `img1-v2`, and so on. Every other non-diagonal entry is a negative for that anchor.

In [ ]:
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_demo, cmap="viridis", vmin=-1, vmax=1)
ax.set_xticks(range(2*N_demo)); ax.set_xticklabels(view_names, rotation=45, ha="right")
ax.set_yticks(range(2*N_demo)); ax.set_yticklabels(view_names)
for r, c in enumerate(pos_target.tolist()):
    ax.add_patch(Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, edgecolor="red", linewidth=2))
ax.set_title("red boxes are positives")
fig.colorbar(im, label="cosine")
plt.tight_layout(); plt.show()

### Step 10 — NT-Xent part D: divide by temperature

Temperature controls how sharp the softmax is. Low temperature magnifies small similarity differences; high temperature makes the distribution flatter.

In [ ]:
tau = 0.5
scaled_demo = sim_demo / tau
scaled_no_self = scaled_demo.clone()
scaled_no_self.fill_diagonal_(-1e9)
print("temperature tau =", tau)
print("cosine range:", round(float(sim_demo.min()), 3), "to", round(float(sim_demo.max()), 3))
finite_scaled = scaled_no_self[scaled_no_self > -1e8]
print("scaled non-self range:", round(float(finite_scaled.min()), 3), "to", round(float(finite_scaled.max()), 3))

**🎛️ Play with it — NT-Xent temperature**

Use the fixed similarity row from the toy batch. **Try:** move `tau` from very low to high. **Watch:** low `tau` puts almost all probability mass on the most similar item; high `tau` spreads mass over negatives. **Why it matters:** temperature sets how aggressively positives must outrank negatives.

In [ ]:
def temperature_play(tau=0.5, anchor=0):
    anchor = int(anchor)
    logits = sim_demo[anchor].clone() / tau
    logits[anchor] = -1e9
    probs = torch.softmax(logits, dim=0)
    colors = ["#ff6b6b" if j == int(pos_target[anchor]) else "#79c0ff" for j in range(len(probs))]
    plt.figure(figsize=(7, 3))
    plt.bar(view_names, probs.detach().numpy(), color=colors)
    plt.xticks(rotation=45, ha="right"); plt.ylabel("softmax probability")
    plt.title("temperature softmax")
    plt.tight_layout(); plt.show()
    print("anchor:", view_names[anchor], "positive:", view_names[int(pos_target[anchor])],
          "positive prob:", round(float(probs[pos_target[anchor]]), 3))

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(temperature_play,
             tau=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05),
             anchor=IntSlider(value=0, min=0, max=2*N_demo-1, step=1))
except Exception:
    temperature_play()

### Step 11 — NT-Xent part E: cross-entropy pulls positives up

For each row, cross-entropy asks the positive partner's logit to win the softmax. Training lowers the loss by making red-box similarities brighter and non-partner similarities darker.

In [ ]:
loss_demo = F.cross_entropy(scaled_no_self, pos_target)
probs_demo = torch.softmax(scaled_no_self, dim=1)
row_df = pd.DataFrame({
    "candidate": view_names,
    "cosine": sim_demo[0].round(decimals=3).tolist(),
    "softmax_prob": probs_demo[0].round(decimals=3).tolist(),
    "role_for_anchor0": ["self removed" if j == 0 else ("positive" if j == int(pos_target[0]) else "negative") for j in range(2*N_demo)],
})
display(row_df)
print("toy NT-Xent loss:", round(float(loss_demo), 4))

idea_after = sim_demo.clone() * 0.35
for r, c in enumerate(pos_target.tolist()):
    idea_after[r, c] = 0.95
idea_after.fill_diagonal_(1.0)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, mat, title in zip(axes, [sim_demo, idea_after], ["before training", "idea after training"]):
    im = ax.imshow(mat, cmap="viridis", vmin=-1, vmax=1)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), label="cosine")
plt.show()

### Step 12 — Package NT-Xent into a reusable function

The function below is the batched version of the tensor walk above: normalize, form the full similarity matrix, blank the diagonal self-match, build partner targets, and call cross-entropy.

In [ ]:
def nt_xent(z, tau=0.5):
    z = F.normalize(z, dim=1)                         # cosine similarity needs unit vectors
    N = z.shape[0] // 2
    sim = z @ z.t() / tau                             # 2N x 2N scaled cosine-similarity matrix
    sim.fill_diagonal_(-1e9)                          # no self-match
    targets = torch.cat([torch.arange(N) + N, torch.arange(N)]).to(z.device)
    return F.cross_entropy(sim, targets)

print("function check on toy batch:", round(float(nt_xent(z_demo, tau=0.5)), 4))

### Step 13 — Define the encoder and projection head

The encoder `f` maps an image to a representation `h`; this is the part we freeze and probe later. The projection head `g` maps `h` to `z`, where contrastive loss is applied; after pretraining, we discard the projection head.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Linear(32, feat)
    def forward(self, x):
        return F.relu(self.fc(self.net(x)))             # h

class ProjHead(nn.Module):
    def __init__(self, fin=64, hid=64, out=32):
        super().__init__()
        self.l1 = nn.Linear(fin, hid)
        self.l2 = nn.Linear(hid, out)
    def forward(self, h):
        return self.l2(F.relu(self.l1(h)))              # z (loss lives here)

### Step 14 — Trace shapes through the components

A small batch of images becomes encoder features `h`, then projection vectors `z`, then one scalar NT-Xent loss. The table is the complete SimCLR data flow for one batch.

In [ ]:
torch.manual_seed(0)
shape_enc, shape_proj = Encoder(), ProjHead()
xb = torch.stack([base(im) for im in imgs[:8]])
with torch.no_grad():
    h = shape_enc(xb)
    z = shape_proj(h)
shape_df = pd.DataFrame([
    ("images", tuple(xb.shape), "two-view input before stacking"),
    ("encoder h", tuple(h.shape), "representation kept for probe"),
    ("projection z", tuple(z.shape), "contrastive-loss space"),
], columns=["stage", "shape", "meaning"])
display(shape_df)

### Step 15 — Parameter table and random-init sanity check

The encoder is small enough to understand. At random initialization, the contrastive loss should be in the ballpark of `ln(2N-1)` because each anchor chooses among `2N-1` non-self candidates.

In [ ]:
enc0, proj0 = Encoder(), ProjHead()
param_rows = []
for module_name, module in [("encoder", enc0), ("projection_head", proj0)]:
    param_rows.append({"component": module_name, "parameters": sum(p.numel() for p in module.parameters())})
param_df = pd.DataFrame(param_rows)
param_df["% of total"] = (100 * param_df["parameters"] / param_df["parameters"].sum()).round(1)
display(param_df)

with torch.no_grad():
    torch.manual_seed(3)
    v1 = torch.stack([aug(imgs[i]) for i in range(16)])
    v2 = torch.stack([aug(imgs[i]) for i in range(16)])
    init_loss = nt_xent(proj0(enc0(torch.cat([v1, v2])))).item()
print("random-init NT-Xent:", round(init_loss, 3), " vs ln(2N-1):", round(float(np.log(31)), 3))

**🎛️ Play with it — batch size means more negatives**

With SimCLR, each anchor has exactly one positive and `2N-2` negatives inside a `2N` view batch. **Try:** change the source-image batch size `N`. **Watch:** negatives grow linearly. **Why it matters:** more negatives usually make contrastive learning harder but richer.

In [ ]:
def batch_negatives(N=128):
    positives = 1
    negatives = 2 * int(N) - 2
    total_choices = 2 * int(N) - 1
    df = pd.DataFrame({"quantity": ["source images N", "positive per anchor", "negatives per anchor", "softmax choices"],
                       "value": [int(N), positives, negatives, total_choices]})
    display(df)
    plt.figure(figsize=(5, 3))
    plt.bar(["positive", "negatives"], [positives, negatives], color=["#ff6b6b", "#79c0ff"])
    plt.title("one positive vs many negatives")
    plt.ylabel("count per anchor")
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(batch_negatives, N=IntSlider(value=128, min=2, max=512, step=2))
except Exception:
    batch_negatives()

### Step 16 — One SimCLR training step

Before the full pretrain loop, watch one optimizer step on a tiny batch. We use a separate demo model so the real run still starts fresh from the original seed.

In [ ]:
torch.manual_seed(0)
demo_enc, demo_proj = Encoder().to(device), ProjHead().to(device)
demo_opt = torch.optim.Adam(list(demo_enc.parameters()) + list(demo_proj.parameters()), lr=1e-3)
bi = list(range(32))
v1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
v2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
before = nt_xent(demo_proj(demo_enc(torch.cat([v1, v2]))))
demo_opt.zero_grad(); before.backward(); demo_opt.step()
after = nt_xent(demo_proj(demo_enc(torch.cat([v1, v2]))))
print(f"one batch NT-Xent: before = {before.item():.3f} -> after one step = {after.item():.3f}")

### Step 17 — Pretrain SimCLR on unlabelled MNIST

Now the real run. Each step samples two augmented views of each image in the batch, pushes them through the encoder and projection head, and minimizes NT-Xent. Labels are not passed into this loop.

In [ ]:
torch.manual_seed(0)
np.random.seed(0)
enc, proj = Encoder().to(device), ProjHead().to(device)
opt = torch.optim.Adam(list(enc.parameters()) + list(proj.parameters()), lr=1e-3)
enc.train(); proj.train()
B = 128
EPOCHS = 15
pretrain_history = []
for ep in range(EPOCHS):
    perm = np.random.permutation(len(imgs))
    tot = 0.0
    nb = 0
    for s in range(0, len(imgs), B):
        bi = perm[s:s + B]
        v1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
        v2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
        loss = nt_xent(proj(enc(torch.cat([v1, v2]))))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    avg = tot / nb
    pretrain_history.append((ep, avg))
    if ep % 3 == 0 or ep == EPOCHS - 1:
        print(f"  pretrain ep {ep}  NT-Xent loss {avg:.3f}")
# Our small CPU run, not a paper number; exact values vary by hardware and seed.

### Step 18 — Freeze the encoder and extract features

Linear evaluation freezes the pretrained encoder. We compute each image's feature vector `h` once; from here on, the probe can only learn a linear boundary on top of these frozen features.

In [ ]:
enc.eval()
for p in enc.parameters():
    p.requires_grad_(False)

feature_batches = []
with torch.no_grad():
    for s in range(0, len(imgs), 256):
        x = torch.stack([base(im) for im in imgs[s:s+256]]).to(device)
        feature_batches.append(enc(x).cpu())
feats = torch.cat(feature_batches)
feat_norm = F.normalize(feats, dim=1)

feat_df = pd.DataFrame({
    "object": ["frozen feature matrix", "labels kept for evaluation"],
    "shape": [tuple(feats.shape), tuple(labels.shape)],
    "used_during_pretrain": [False, False],
})
display(feat_df)

### Step 19 — Train the linear probe and from-scratch baselines

For each label budget, `linear_probe` trains only a linear classifier on frozen features. `from_scratch` trains a fresh encoder+classifier end-to-end on the same few labels. We average over the original three seeds.

In [ ]:
def split_for_budget(seed, n_lab):
    g = np.random.RandomState(seed)
    perm = g.permutation(len(labels))
    te_size = min(600, max(20, len(labels) // 3))
    n_eff = min(int(n_lab), len(labels) - te_size)
    return perm[:n_eff], perm[-te_size:]

def linear_probe(n_lab):
    accs = []
    for seed in range(3):
        tr, te = split_for_budget(seed, n_lab)
        torch.manual_seed(seed)
        clf = nn.Linear(feats.shape[1], 10)
        o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad()
            F.cross_entropy(clf(feats[tr]), labels[tr]).backward()
            o.step()
        with torch.no_grad():
            accs.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

def from_scratch(n_lab):
    accs = []
    for seed in range(3):
        torch.manual_seed(seed)
        tr, te = split_for_budget(seed, n_lab)
        net = nn.Sequential(Encoder(), nn.Linear(64, 10))
        o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr])
        net.train()
        for _ in range(60):
            o.zero_grad()
            F.cross_entropy(net(Xtr), labels[tr]).backward()
            o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            accs.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

budgets = [20, 50, 100, 300]
results_dict = {}
print("\nlabels | probe(frozen SimCLR) | from-scratch | probe wins?")
for n in budgets:
    p, s = linear_probe(n), from_scratch(n)
    results_dict[n] = {"probe": p, "from_scratch": s, "gap": p - s}
    print(f"{n:6d} |        {p:.3f}         |    {s:.3f}     |   {'YES' if p > s else 'no'}")
# Headline: in our small run, the frozen-SimCLR probe should be strongest in the low-label regime.
# Exact numbers vary by hardware/seed; these are our run, not the papers' reported results.

## Visualize the data & results

_Does the finished system work: in the low-label regime, does a linear probe on frozen contrastive features beat a from-scratch classifier, and do frozen features show digit structure even though pretraining used no labels?_

### Step 20 — Pretraining loss curve

The recorded NT-Xent loss is noisy because every epoch samples new random views, but it should trend downward as positives become easier to identify among negatives.

In [ ]:
loss_df = pd.DataFrame(pretrain_history, columns=["epoch", "nt_xent_loss"])
display(loss_df)
plt.figure(figsize=(7, 3.4))
plt.plot(loss_df["epoch"], loss_df["nt_xent_loss"], marker="o", color="#7ee787")
plt.xlabel("epoch"); plt.ylabel("NT-Xent loss")
plt.title("SimCLR pretrain loss")
plt.show()

### Step 21 — Headline plot: probe vs from-scratch

This table and line plot use the already-computed `results_dict`; no retraining happens here. The comparison is labelled **our run** because it is this tiny CPU teaching setup, not a reported paper benchmark.

In [ ]:
result_df = pd.DataFrame([
    {"labels": n, "probe_frozen_simclr": v["probe"], "from_scratch": v["from_scratch"], "probe_minus_scratch": v["gap"]}
    for n, v in results_dict.items()
])
display(result_df.round(3))

plt.figure(figsize=(7, 3.6))
plt.plot(result_df["labels"], result_df["probe_frozen_simclr"], marker="o", label="frozen SimCLR probe", color="#79c0ff")
plt.plot(result_df["labels"], result_df["from_scratch"], marker="o", label="from scratch", color="#ff6b6b")
plt.xlabel("label budget"); plt.ylabel("accuracy")
plt.ylim(0, 1); plt.title("our run: probe vs scratch")
plt.legend(); plt.show()

**🎛️ Play with it — label-budget slider**

The expensive training is already done. **Try:** choose a label budget. **Watch:** the probe and from-scratch bars update from `results_dict` only. **Why it matters:** pretrained features are most valuable when labels are scarce.

In [ ]:
def budget_play(labels_budget=20):
    n = int(labels_budget)
    vals = results_dict[n]
    plt.figure(figsize=(5, 3))
    plt.bar(["probe", "scratch"], [vals["probe"], vals["from_scratch"]], color=["#79c0ff", "#ff6b6b"])
    plt.ylim(0, 1); plt.ylabel("accuracy")
    plt.title(f"budget {n}: already computed")
    plt.show()
    print(f"probe={vals['probe']:.3f}  scratch={vals['from_scratch']:.3f}  gap={vals['gap']:.3f}")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(budget_play, labels_budget=Dropdown(options=budgets, value=20))
except Exception:
    budget_play()

### Step 22 — Frozen-feature PCA scatter

Now we color the frozen features by the true digit label **after** pretraining. If self-supervised learning found useful structure, digits should form visible neighborhoods even though labels were never used in NT-Xent.

In [ ]:
scatter_idx = np.random.RandomState(1).permutation(len(labels))[:min(1000, len(labels))]
X = feats[scatter_idx].numpy()
X = X - X.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
coords = X @ Vt[:2].T

plt.figure(figsize=(6.5, 5.2))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=labels[scatter_idx].numpy(), cmap="tab10", s=12, alpha=0.8)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("frozen features PCA")
plt.colorbar(sc, label="true digit label")
plt.show()

**🎛️ Play with it — image, two views, nearest neighbour**

Pick an image and compare its two random augmented views with its nearest neighbour in frozen feature space. **Try:** several digits. **Watch:** neighbours often share visual structure or class. **Why it matters:** this is geometry learned without labels.

In [ ]:
def nearest_play(image_id=0):
    image_id = int(np.clip(image_id, 0, len(imgs) - 1))
    sims = feat_norm @ feat_norm[image_id]
    sims[image_id] = -2
    nn_id = int(torch.argmax(sims))
    torch.manual_seed(10)
    a = aug(imgs[image_id])
    torch.manual_seed(11)
    b = aug(imgs[image_id])
    fig, axes = plt.subplots(1, 4, figsize=(8, 2.4))
    panels = [(base(imgs[image_id]), "original"), (a, "view 1"), (b, "view 2"), (base(imgs[nn_id]), f"nearest {nn_id}")]
    for ax, (tensor, title) in zip(axes, panels):
        ax.imshow(tensor.squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title); ax.axis("off")
    plt.suptitle("views and feature neighbour")
    plt.tight_layout(); plt.show()
    print("chosen label:", int(labels[image_id]), "nearest label:", int(labels[nn_id]),
          "cosine:", round(float(sims[nn_id]), 3))

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(nearest_play, image_id=IntSlider(value=0, min=0, max=len(imgs)-1, step=1))
except Exception:
    nearest_play()

**🎛️ Play with it — cosine similarity of two frozen features**

Pick two images and read their feature cosine similarity. **Try:** same-digit pairs and different-digit pairs. **Watch:** higher cosine means the frozen encoder places them closer. **Why it matters:** the linear probe succeeds only if this geometry makes classes easy to separate.

In [ ]:
def pair_similarity(i=0, j=1):
    i = int(np.clip(i, 0, len(imgs) - 1)); j = int(np.clip(j, 0, len(imgs) - 1))
    sim = float(feat_norm[i] @ feat_norm[j])
    fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.3))
    for ax, k in zip(axes, [i, j]):
        ax.imshow(imgs[k], cmap="gray")
        ax.set_title(f"id {k}, y={int(labels[k])}"); ax.axis("off")
    plt.suptitle("feature cosine")
    plt.tight_layout(); plt.show()
    print(f"cosine(feature[{i}], feature[{j}]) = {sim:.3f}")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(pair_similarity,
             i=IntSlider(value=0, min=0, max=len(imgs)-1, step=1),
             j=IntSlider(value=1, min=0, max=len(imgs)-1, step=1))
except Exception:
    pair_similarity()

### Step 23 — What to remember

SimCLR never sees labels during pretraining. It sees only two augmented views of the same image as a positive pair, all other views as negatives, and NT-Xent as the pressure that makes positives similar. The frozen encoder can then support a cheap linear probe, especially when labelled examples are scarce.